# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

**Dataset:** data cuaca historis per jam 34 ibu kota provinsi Indonesia (Open-Meteo, reanalisis ERA5), periode 2015-2025. Kunci unik data: `kota` + `time`.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'cuaca_indonesia_2015_2025.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cuaca_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 3,278,688 baris x 8 kolom


time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,kota,lat,lon
str,f64,i64,f64,f64,str,f64,f64
"""2015-01-01T00:00""",27.3,82,0.0,16.5,"""Ambon""",-3.7,128.18
"""2015-01-01T01:00""",27.3,82,0.0,17.4,"""Ambon""",-3.7,128.18
"""2015-01-01T02:00""",27.2,81,0.2,16.6,"""Ambon""",-3.7,128.18
"""2015-01-01T03:00""",27.3,80,0.0,17.0,"""Ambon""",-3.7,128.18
"""2015-01-01T04:00""",27.4,80,0.0,19.4,"""Ambon""",-3.7,128.18


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""time""","""VARCHAR""","""2015-01-01T00:00""","""2025-12-31T23:00""",112872,null,null,null,null,null,3278688,0.00
"""temperature_2m""","""DOUBLE""","""13.9""","""43.0""",243,"""26.434156314965165""","""2.3527682301919555""","""24.776344438329318""","""26.082755618763482""","""27.971765527113345""",3278688,0.00
"""relative_humidity_2m""","""BIGINT""","""13""","""100""",82,"""84.95728535316566""","""11.95327938094196""","""78""","""88""","""94""",3278688,0.00
"""precipitation""","""DOUBLE""","""0.0""","""61.6""",394,"""0.3282979350278918""","""1.089138285822609""","""0.0""","""0.0""","""0.1949851289674499""",3278688,0.00
"""wind_speed_10m""","""DOUBLE""","""0.0""","""58.6""",392,"""6.815683590509827""","""4.440079262285669""","""3.662466034752867""","""5.6829038404126955""","""8.852879157662159""",3278688,0.00
"""kota""","""VARCHAR""","""Ambon""","""Yogyakarta""",29,null,null,null,null,null,3278688,0.00
"""lat""","""DOUBLE""","""-10.18""","""5.55""",33,"""-2.8017647058910287""","""3.737286838142096""","""-6.12""","""-2.6045353641467126""","""-0.03""",3278688,0.00
"""lon""","""DOUBLE""","""95.32""","""140.72""",36,"""113.6114705884034""","""10.385927335549443""","""105.26""","""112.75""","""119.93802736175552""",3278688,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64


## 3. Penanganan Missing Values

Hasil bagian 2 menunjukkan dataset ini tidak memiliki missing values, jadi tidak ada baris yang dihapus atau diimputasi. Langkah yang dilakukan adalah memperbaiki tipe data: kolom `time` yang masih berupa String dikonversi menjadi Datetime agar dapat dipakai pada analisis temporal (Milestone 3).

Sebagai pengaman, jika ada null pada kolom numerik, nilainya diisi dengan median per kota (data cuaca bergantung pada lokasi, sehingga median global kurang tepat). Langkah ini tidak berpengaruh bila tidak ada null.

In [4]:
KOLOM_FLOAT = ['temperature_2m', 'precipitation', 'wind_speed_10m']

df_missing_handled = df.with_columns(
    # Konversi time dari String ke Datetime
    pl.col('time').str.to_datetime('%Y-%m-%dT%H:%M'),
    # Isi null numerik dengan median per kota (pengaman)
    *[pl.col(k).fill_null(pl.col(k).median().over('kota')) for k in KOLOM_FLOAT],
    pl.col('relative_humidity_2m').fill_null(
        pl.col('relative_humidity_2m').median().over('kota').round(0).cast(pl.Int64)
    ),
)

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')
print(f'Tipe kolom time: {df_missing_handled.schema["time"]}')

Null sebelum: 0
Null sesudah: 0
Tipe kolom time: Datetime(time_unit='us', time_zone=None)


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

- **Suhu dan kecepatan angin:** IQR + windsorization (clip ke batas). Hasil clip disimpan pada kolom baru `*_winsor`, sedangkan nilai asli tetap dipertahankan karena suhu tinggi dan angin kencang adalah kejadian cuaca yang nyata.
- **Curah hujan:** tidak dipotong dengan IQR. Distribusinya sangat miring (median 0), sehingga batas IQR akan memotong semua kejadian hujan lebat yang nyata. Hujan hanya ditandai lewat ambang domain (`hujan_ekstrem`).

In [5]:
KOLOM_NUMERIK = ['temperature_2m', 'wind_speed_10m']  # IQR + windsorization
KOLOM_HUJAN = 'precipitation'
AMBANG_HUJAN_EKSTREM = 20.0  # mm/jam (kategori hujan sangat lebat)

batas = {}
for kolom in KOLOM_NUMERIK:
    q1 = df_missing_handled[kolom].quantile(0.25)
    q3 = df_missing_handled[kolom].quantile(0.75)
    iqr = q3 - q1
    batas[kolom] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    print(f'{kolom}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}')
    print(f'  Batas bawah: {batas[kolom][0]:.2f}, Batas atas: {batas[kolom][1]:.2f}')

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    *[
        pl.col(k).clip(lower_bound=b[0], upper_bound=b[1]).alias(f'{k}_winsor')
        for k, b in batas.items()
    ],
    *[
        ((pl.col(k) < b[0]) | (pl.col(k) > b[1])).alias(f'outlier_{k}')
        for k, b in batas.items()
    ],
    (pl.col(KOLOM_HUJAN) > AMBANG_HUJAN_EKSTREM).alias('hujan_ekstrem'),
)

jumlah_outlier = df_outlier_handled.select(
    [
        pl.col(c).sum().alias(c)
        for c in df_outlier_handled.columns
        if c.startswith('outlier_') or c == 'hujan_ekstrem'
    ]
)
jumlah_outlier

temperature_2m: Q1=24.80, Q3=28.00, IQR=3.20
  Batas bawah: 20.00, Batas atas: 32.80
wind_speed_10m: Q1=3.70, Q3=8.90, IQR=5.20
  Batas bawah: -4.10, Batas atas: 16.70


outlier_temperature_2m,outlier_wind_speed_10m,hujan_ekstrem
u32,u32,u32
33610,128064,487


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

Pada dataset ini satu baris mewakili satu kota pada satu jam, sehingga kunci uniknya adalah gabungan `kota` + `time`. Setelah itu dicek juga kelengkapan data: setiap kota seharusnya memiliki jumlah jam yang sama.

In [6]:
KOLOM_KUNCI = ['kota', 'time']

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled.select(KOLOM_KUNCI).n_unique()
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep='first',
    maintain_order=True,
)

# Kelengkapan: jumlah jam per kota harus sama dengan rentang waktu penuh
jam_penuh = int((df_dedup['time'].max() - df_dedup['time'].min()).total_seconds() // 3600) + 1
kelengkapan = df_dedup.group_by('kota').len().sort('kota')
print(f'Jumlah kota: {kelengkapan.height}')
print(f'Jam per kota (min/maks): {kelengkapan["len"].min():,} / {kelengkapan["len"].max():,} (diharapkan {jam_penuh:,})')

Total baris: 3,278,688
Baris unik berdasarkan ['kota', 'time']: 3,278,688
Duplikat: 0
Jumlah kota: 34
Jam per kota (min/maks): 96,432 / 96,432 (diharapkan 96,432)


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

Aturan yang dipakai: tidak ada null, nilai winsorized berada dalam batas IQR, kelembapan 0-100%, curah hujan dan kecepatan angin tidak negatif, suhu 0-50 °C, dan koordinat berada dalam wilayah Indonesia.

In [7]:
connection.register('dataset_bersih', df_dedup)

suhu_bawah, suhu_atas = batas['temperature_2m']
angin_bawah, angin_atas = batas['wind_speed_10m']

validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) FILTER (WHERE temperature_2m IS NULL OR relative_humidity_2m IS NULL
                           OR precipitation IS NULL OR wind_speed_10m IS NULL) AS pelanggaran_null,
        COUNT(*) FILTER (WHERE temperature_2m_winsor < {suhu_bawah} OR temperature_2m_winsor > {suhu_atas}
                           OR wind_speed_10m_winsor < {angin_bawah} OR wind_speed_10m_winsor > {angin_atas}) AS pelanggaran_batas,
        COUNT(*) FILTER (WHERE relative_humidity_2m < 0 OR relative_humidity_2m > 100) AS langgar_kelembapan,
        COUNT(*) FILTER (WHERE precipitation < 0) AS langgar_hujan,
        COUNT(*) FILTER (WHERE wind_speed_10m < 0) AS langgar_angin,
        COUNT(*) FILTER (WHERE temperature_2m < 0 OR temperature_2m > 50) AS langgar_suhu,
        COUNT(*) FILTER (WHERE lat NOT BETWEEN -11.5 AND 6.5 OR lon NOT BETWEEN 94.5 AND 141.5) AS langgar_koordinat
    FROM dataset_bersih
""").pl()

pelanggaran = validasi.drop('total_baris').row(0)
print('Semua aturan validasi lolos' if sum(pelanggaran) == 0 else 'Ada pelanggaran, periksa kolom pada tabel di bawah')
validasi

Semua aturan validasi lolos


total_baris,pelanggaran_null,pelanggaran_batas,langgar_kelembapan,langgar_hujan,langgar_angin,langgar_suhu,langgar_koordinat
i64,i64,i64,i64,i64,i64,i64,i64
3278688,0,0,0,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning. Setelah disimpan, file hasilnya diprofilkan ulang dengan `SUMMARIZE` untuk memastikan isinya sesuai.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Baris berubah: {len(df) - len(df_dedup):,}')
print(f'Ukuran file: {OUTPUT_PATH.stat().st_size / 1_000_000:.1f} MB')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 3,278,688
Baris sesudah cleaning: 3,278,688
Baris berubah: 0
Ukuran file: 37.5 MB
Data bersih tersimpan di: /home/jovyan/work/data/processed/cuaca_bersih.parquet


In [9]:
profil_bersih = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [str(OUTPUT_PATH)]
).pl()
profil_bersih

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""time""","""TIMESTAMP""","""2015-01-01 00:00:00""","""2025-12-31 23:00:00""",88398,"""2020-07-01 23:30:00""",null,"""2017-10-01 18:45:17.616183""","""2020-06-29 01:27:29.631048""","""2023-03-30 01:09:22.778635""",3278688,0.00
"""temperature_2m""","""DOUBLE""","""13.9""","""43.0""",243,"""26.434156314965136""","""2.3527682301919777""","""24.779098742101706""","""26.077765453973583""","""27.974107138750806""",3278688,0.00
"""relative_humidity_2m""","""BIGINT""","""13""","""100""",82,"""84.95728535316566""","""11.953279380941813""","""78""","""88""","""94""",3278688,0.00
"""precipitation""","""DOUBLE""","""0.0""","""61.6""",394,"""0.32829793502790255""","""1.0891382858226144""","""0.0""","""0.0""","""0.19511373243308022""",3278688,0.00
"""wind_speed_10m""","""DOUBLE""","""0.0""","""58.6""",392,"""6.815683590509616""","""4.440079262285677""","""3.663451290034299""","""5.685104610373972""","""8.848869255087331""",3278688,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""temperature_2m_winsor""","""DOUBLE""","""20.0""","""32.8""",110,"""26.43336566333956""","""2.3199115465925018""","""24.779098742101706""","""26.077765453973583""","""27.974107138750806""",3278688,0.00
"""wind_speed_10m_winsor""","""DOUBLE""","""0.0""","""16.700000000000003""",169,"""6.683733218898508""","""4.034360275582002""","""3.663451290034299""","""5.685104610373972""","""8.848869255087331""",3278688,0.00
"""outlier_temperature_2m""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,3278688,0.00


## Catatan Milestone 2

D## Catatan Milestone 2

- **Dimensi kualitas yang paling bermasalah:** tipe data dan distribusi nilai. Kolom `time` masih bertipe String. Kecepatan angin punya outlier paling banyak (128.064 baris, 3,91%), dan curah hujan sangat miring (median 0, maksimum 61,6 mm/jam). Kelengkapan dan keunikan data sudah baik: tidak ada missing values dan tidak ada duplikat.
- **Missing values:** 0% null di seluruh kolom, jadi tidak ada baris yang dihapus atau diimputasi. Kolom `time` dikonversi dari String ke Datetime. Pengaman median per kota disiapkan jika ada null, tetapi tidak terpakai.
- **Outlier:** suhu dan kecepatan angin memakai IQR + windsorization (clip ke batas, kolom baru `*_winsor`, nilai asli dipertahankan). Batas suhu 20,00–32,80 °C (33.610 baris terdeteksi, 1,03%) dan batas angin -4,10–16,70 (128.064 baris, 3,91%). Karena angin tidak mungkin negatif, batas bawah efektifnya 0. Curah hujan tidak dipotong dengan IQR, hanya ditandai `hujan_ekstrem` bila > 20 mm/jam (487 baris, 0,015%), karena IQR akan memotong kejadian hujan lebat yang nyata.
- **Duplikat:** 0 baris duplikat berdasarkan kunci (`kota`, `time`). Kelengkapan: 34 kota × 96.432 jam per kota, semua kota lengkap.
- **Jumlah baris:** sebelum 3.278.688, sesudah 3.278.688, berubah 0 baris. Cleaning hanya mengubah tipe data dan menambah kolom (`*_winsor` dan flag outlier), tanpa menghapus baris.
- **Aturan validasi:** tidak ada null; nilai winsorized berada di dalam batas IQR; kelembapan 0–100%; hujan dan angin tidak negatif; suhu 0–50 °C; koordinat di wilayah Indonesia. Hasil: 0 pelanggaran untuk semua aturan.
- **Catatan:** data adalah reanalisis ERA5 (model grid), sehingga nilai ekstrem perlu diperlakukan sebagai estimasi model, bukan observasi stasiun. Hasil disimpan di `data/processed/cuaca_bersih.parquet` (37,5 MB).